# A1.1 · The reference architecture for agentic AI

**Function A — Security Architecture & Platform → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Both directions*

| | |
|---|---|
| Open-source tooling | kagent, OpenTelemetry |
| Open-weight models | Llama 3.3, GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Every risk in this chapter and every control in the next two names a part of
one picture. This is that picture, and it is deliberately vendor-neutral —
these components exist under different product names in every agent platform,
and the risks attach to the component, not to the brand.

**Ingress.** Where a request enters: a chat surface, an API call, a webhook, a
scheduled trigger, another system. It carries the requester's identity and
whatever text they supplied.

**Orchestrator.** Decides which agent handles what, and in what topology. In a
single-agent system this is a few lines; in a multi-agent one it is the thing
that holds the whole design.

**Agent runtime.** The loop: plan, call a tool, observe the result, decide
again, stop. This is the component that turns text into consequence.

**Model.** Predicts tokens. Holds no credential, opens no socket, changes
nothing. Most of what people fear "the model doing" is done by the runtime.

**Tools and MCP servers.** The only components that change anything. An MCP
server is a third party's process whose tool descriptions land in your context.

**Knowledge and memory.** Retrieval pulls documents in at query time; memory
persists state across turns and sessions. Both inject text the user did not
write.

**Messaging.** The agent-to-agent channel in a multi-agent topology.

**Identity and policy.** Who is calling, on whose behalf, and whether this call
is permitted. These wrap every other component.

**Egress.** Where data is allowed to go — the last boundary before it leaves.

**Observability.** What can be reconstructed afterwards.

These compose into five topologies, and the topology decides which risks apply:
**single agent**, **orchestrator–worker**, **peer handoff**, **swarm**, and
**workflow with agent steps**.

## 2 · The architecture, and the five topologies it composes into\n\n`trust` is how much authority content originating at that component should carry: 2 is the authenticated requester, 1 is machinery with no authority of its own, 0 is anything an outsider can write into.

In [ ]:
COMPONENTS = {
 "ingress":       ("where a request enters the system",                 2),
 "orchestrator":  ("routes work to agents and chooses the topology",    2),
 "agent_runtime": ("the plan-act-observe loop; turns text into action", 2),
 "model":         ("predicts tokens; no credential, no socket",         1),
 "tools":         ("the only components that change anything",          2),
 "mcp":           ("a third party's process exposing tools",            0),
 "knowledge":     ("retrieval; pulls documents in at query time",       0),
 "memory":        ("state that persists across turns and sessions",     1),
 "messaging":     ("the agent-to-agent channel",                        1),
 "identity":      ("who is calling, and on whose behalf",               2),
 "policy":        ("may this caller do this, to this resource",         2),
 "egress":        ("where data is allowed to go",                       2),
 "observability": ("what can be reconstructed afterwards",              2),
}

TOPOLOGIES = {
 "single agent":            ["ingress", "agent_runtime", "tools"],
 "orchestrator-worker":     ["ingress", "orchestrator", "agent_runtime", "messaging", "tools"],
 "peer handoff":            ["ingress", "agent_runtime", "messaging", "agent_runtime", "tools"],
 "swarm":                   ["ingress", "orchestrator", "messaging", "agent_runtime", "memory", "tools"],
 "workflow with agent steps":["ingress", "orchestrator", "agent_runtime", "tools", "policy"],
}

print(f"{'component':15s}{'trust':>6}  role")
for name in sorted(COMPONENTS):
    role, trust = COMPONENTS[name]
    print(f"{name:15s}{trust:>6}  {role}")

print("\nUNTRUSTED SOURCES (trust 0) - an outsider can author content here")
print("   " + ", ".join(sorted(c for c, (_, t) in COMPONENTS.items() if t == 0)))

print("\nTOPOLOGIES")
for name in sorted(TOPOLOGIES):
    hops = TOPOLOGIES[name]
    print(f"   {name:26s}{' -> '.join(hops)}")

# the boundary every risk in this chapter crosses, in every topology
print("\nIn all five topologies the same edge exists: something reaches")
print("agent_runtime, and agent_runtime reaches tools. That edge is where text")
print("becomes consequence, and it is the edge every control in Chapters 2 and 3")
print("is trying to stand on.")
assert all("agent_runtime" in h and "tools" in h for h in TOPOLOGIES.values())

## What you just proved

Thirteen components print with the authority their content should carry, three of them at trust 0 — mcp, knowledge and the corpus behind it — and five topologies print as component chains. Every topology contains the same edge: agent_runtime reaching tools.

## Your turn

Draw these thirteen components for one agentic system you run, and mark which topology it is. The useful output is the list of trust-0 components you actually have, because that list is the input surface for the next fifteen lessons.

---

**Next → [A1.2 · Prompt injection](https://spbreed.github.io/cyber-commons/lessons/A1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*